# Hangman

A Hangman game using the google-10000-english word list.
Run the cells from top to bottom. The last cell starts the game and asks for your guesses.

In [1]:
import random
import string
import urllib.request
from pathlib import Path

WORDS_URL = (
    "https://raw.githubusercontent.com/first20hours/google-10000-english/"
    "master/google-10000-english-no-swears.txt"
)
WORDS_FILE = Path("words.txt")   # optional local copy (one word per line)
MIN_LENGTH = 4                   # shorter words make for a dull game
MAX_MISSES = 6

## Load the words

In [2]:
def load_words():
    """Download the word list; fall back to a local words.txt if offline."""
    try:
        with urllib.request.urlopen(WORDS_URL, timeout=10) as resp:
            text = resp.read().decode("utf-8")
    except OSError:
        if not WORDS_FILE.exists():
            raise RuntimeError(
                "Could not download the word list and no words.txt was found. "
                "Put a words.txt next to this notebook and run again."
            )
        text = WORDS_FILE.read_text()
    words = [w.strip().lower() for w in text.splitlines()]
    return [w for w in words if w.isalpha() and len(w) >= MIN_LENGTH]

words = load_words()
print(f"{len(words)} words loaded.")

8810 words loaded.


## Game pieces

In [3]:
STAGES = [
    """
  +---+
  |   |
      |
      |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
      |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
  |   |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|   |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
 /    |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
 / \\  |
      |
=========""",
]


def display_word(word, guessed):
    return " ".join(c if c in guessed else "_" for c in word)


def get_guess(guessed):
    """Prompt until the player enters a single new letter."""
    while True:
        raw = input("Guess a letter: ").strip().lower()
        if len(raw) != 1 or raw not in string.ascii_lowercase:
            print("Please enter a single letter (a-z).")
        elif raw in guessed:
            print(f"You already guessed '{raw}'.")
        else:
            return raw

## One round

In [4]:
def play_round(word):
    """Play one game. Returns True if the player won."""
    guessed = set()
    misses = 0

    while misses < MAX_MISSES:
        print(STAGES[misses])
        print(f"\nWord:   {display_word(word, guessed)}")
        wrong = sorted(g for g in guessed if g not in word)
        print(f"Missed: {' '.join(wrong) if wrong else '-'}  "
              f"({MAX_MISSES - misses} left)")

        letter = get_guess(guessed)
        guessed.add(letter)

        if letter in word:
            print(f"Yes! '{letter}' is in the word.")
            if all(c in guessed for c in word):
                print(STAGES[misses])
                print(f"\nWord:   {display_word(word, guessed)}")
                return True
        else:
            misses += 1
            print(f"No '{letter}' in the word.")

    print(STAGES[MAX_MISSES])
    print(f"\nOut of guesses! The word was '{word}'.")
    return False

## Play
Run this cell to start. It keeps score and asks if you want another round.

In [ ]:
def main():
    wins = losses = 0
    print("=== HANGMAN ===")
    print(f"You get {MAX_MISSES} wrong guesses.")

    while True:
        if play_round(random.choice(words)):
            wins += 1
            print("You won!")
        else:
            losses += 1
        print(f"Score - wins: {wins}, losses: {losses}")

        if input("\nPlay again? (y/n): ").strip().lower() != "y":
            print("Thanks for playing!")
            break


main()

=== HANGMAN ===
You get 6 wrong guesses.

  +---+
  |   |
      |
      |
      |
      |

Word:   _ _ _ _ _ _ _
Missed: -  (6 left)


Guess a letter:  kruthik


Please enter a single letter (a-z).


Guess a letter:  k


No 'k' in the word.

  +---+
  |   |
  O   |
      |
      |
      |

Word:   _ _ _ _ _ _ _
Missed: k  (5 left)


...

No 'p' in the word.

  +---+
  |   |
  O   |
 /|   |
      |
      |

Word:   _ _ _ u r _ _
Missed: i k p  (3 left)


Guess a letter:  wen


Please enter a single letter (a-z).


Guess a letter:  a


Yes! 'a' is in the word.

  +---+
  |   |
  O   |
 /|   |
      |
      |

Word:   a _ _ u r _ _
Missed: i k p  (3 left)
